# Tabela Gold — `ecommerce_enderecos`

Este notebook aplica regras de qualidade usando **PySpark**, separa registros válidos e rejeitados, e envia ambos para tabelas separadas no SQL Server.

Fontes:
- `squad1.dq_monitoring_logs`
- `squad1.silver_ecommerce_enderecos`

Saídas Delta:
- `squad1.gold_dq_resumo_por_regra_enderecos`
- `squad1.gold_dq_resumo_por_tabela_enderecos`
- `squad1.gold_enderecos_kpis_geral`
- `squad1.gold_enderecos_kpis_por_uf`
- `squad1.gold_enderecos_kpis_por_apelido`

As tabelas Gold são gravadas como Delta usando `saveAsTable`.  
A réplica para SQL Server é opcional e executada somente se as variáveis JDBC estiverem carregadas pelo `config`.

In [0]:
# Execute o config antes deste notebook se precisar replicar as Gold para SQL Server:
# %run ../config/config

from pyspark.sql import functions as F
from pyspark.sql import DataFrame
from functools import reduce


## 1. Parâmetros

In [0]:
DQ_LOGS_TABLE = "squad1.dq_monitoring_logs"
SILVER_ENDERECOS_TABLE = "squad1.silver_ecommerce_enderecos"

GOLD_DQ_REGRA_ENDERECOS_TABLE = "squad1.gold_dq_resumo_por_regra_enderecos"
GOLD_DQ_TABELA_ENDERECOS_TABLE = "squad1.gold_dq_resumo_por_tabela_enderecos"

GOLD_KPI_GERAL_TABLE = "squad1.gold_enderecos_kpis_geral"
GOLD_KPI_UF_TABLE = "squad1.gold_enderecos_kpis_por_uf"
GOLD_KPI_APELIDO_TABLE = "squad1.gold_enderecos_kpis_por_apelido"

NOME_TABELA_SILVER = "silver_ecommerce_enderecos"


## 2. Funções auxiliares

In [0]:
def garantir_tabela_existe(nome_tabela: str):
    if not spark.catalog.tableExists(nome_tabela):
        raise Exception(f"Tabela não encontrada: {nome_tabela}")
    print(f"Tabela encontrada: {nome_tabela}")


def salvar_delta_gold(df: DataFrame, nome_tabela: str):
    (
        df.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(nome_tabela)
    )
    print(f"Gold Delta criada/atualizada: {nome_tabela}")


def variaveis_sqlserver_disponiveis() -> bool:
    obrigatorias = [
        "JDBC_HOSTNAME",
        "JDBC_DATABASE",
        "JDBC_USERNAME",
        "JDBC_PASSWORD"
    ]
    return all(v in globals() and globals()[v] not in [None, ""] for v in obrigatorias)


def replicar_sqlserver_gold(df: DataFrame, tabela_sql: str):
    """Replica uma tabela Gold agregada para SQL Server.
    Usa overwrite porque Gold é agregada/recalculada, evitando duplicidade.
    """
    if not variaveis_sqlserver_disponiveis():
        print(f"Variáveis JDBC não carregadas. Pulando réplica SQL Server de {tabela_sql}.")
        return

    (
        df.write
        .format("sqlserver")
        .mode("overwrite")
        .option("host", JDBC_HOSTNAME)
        .option("port", "1433")
        .option("database", JDBC_DATABASE)
        .option("user", JDBC_USERNAME)
        .option("password", JDBC_PASSWORD)
        .option("dbtable", f"squad1.{tabela_sql}")
        .option("encrypt", "true")
        .option("trustServerCertificate", "false")
        .save()
    )

    print(f"Tabela replicada no SQL Server: squad1.{tabela_sql}")


## 3. Validar fontes

In [0]:
garantir_tabela_existe(DQ_LOGS_TABLE)
garantir_tabela_existe(SILVER_ENDERECOS_TABLE)

df_logs = spark.table(DQ_LOGS_TABLE)
df_silver_enderecos = spark.table(SILVER_ENDERECOS_TABLE)

print("Schema dq_monitoring_logs:")
df_logs.printSchema()

print("Schema silver_ecommerce_enderecos:")
df_silver_enderecos.printSchema()


## 4. Gold 1 — % de falha por regra por dia

In [0]:
df_logs_enderecos = (
    df_logs
    .filter(F.col("tabela") == F.lit(NOME_TABELA_SILVER))
    .withColumn("data_referencia", F.to_date(F.col("timestamp_execucao")))
)

df_gold_dq_resumo_por_regra_enderecos = (
    df_logs_enderecos
    .groupBy(
        "data_referencia",
        "tabela",
        "regra",
        "severidade"
    )
    .agg(
        F.sum("qtd_registros_falhos").cast("int").alias("qtd_registros_falhos"),
        F.sum("qtd_registros_total").cast("int").alias("qtd_registros_total"),
        F.countDistinct("arquivo_origem").alias("qtd_arquivos_avaliados"),
        F.max("timestamp_execucao").alias("gold_updated_at")
    )
    .withColumn(
        "pct_falha",
        F.when(
            F.col("qtd_registros_total") > 0,
            F.round((F.col("qtd_registros_falhos") / F.col("qtd_registros_total")) * 100, 4)
        ).otherwise(F.lit(0.0))
    )
    .withColumn(
        "status_geral",
        F.when(F.col("qtd_registros_falhos") > 0, F.lit("FAIL"))
         .otherwise(F.lit("PASS"))
    )
)

display(df_gold_dq_resumo_por_regra_enderecos)


## 5. Gold 2 — % de registros limpos por tabela por hora

In [0]:
# Detecta automaticamente as colunas booleanas de falha criadas na Silver.
colunas_falha = [
    c for c in df_silver_enderecos.columns
    if c.startswith("r") and c.endswith("_falhou")
]

if len(colunas_falha) == 0:
    raise Exception("Nenhuma coluna de falha encontrada na Silver. Esperado padrão: r*_falhou")

print("Colunas de falha encontradas:")
for c in colunas_falha:
    print("-", c)

condicao_tem_falha = reduce(
    lambda a, b: a | b,
    [F.coalesce(F.col(c).cast("boolean"), F.lit(False)) for c in colunas_falha]
)

df_silver_enderecos_flags = (
    df_silver_enderecos
    .withColumn("tem_falha", condicao_tem_falha)
    .withColumn("linha_limpa", ~F.col("tem_falha"))
    .withColumn(
        "hora_referencia",
        F.date_trunc(
            "hour",
            F.coalesce(
                F.col("silver_processed_at"),
                F.col("bronze_ingested_at"),
                F.current_timestamp()
            )
        )
    )
)

df_gold_dq_resumo_por_tabela_enderecos = (
    df_silver_enderecos_flags
    .groupBy("hora_referencia")
    .agg(
        F.lit(NOME_TABELA_SILVER).alias("tabela"),
        F.count("*").cast("int").alias("qtd_registros_total"),
        F.sum(F.when(F.col("tem_falha"), 1).otherwise(0)).cast("int").alias("qtd_registros_com_falha"),
        F.sum(F.when(F.col("linha_limpa"), 1).otherwise(0)).cast("int").alias("qtd_registros_limpos"),
        F.countDistinct("bronze_source_file").alias("qtd_arquivos_avaliados"),
        F.current_timestamp().alias("gold_updated_at")
    )
    .withColumn(
        "pct_registros_limpos",
        F.when(
            F.col("qtd_registros_total") > 0,
            F.round((F.col("qtd_registros_limpos") / F.col("qtd_registros_total")) * 100, 4)
        ).otherwise(F.lit(0.0))
    )
    .withColumn(
        "pct_registros_com_falha",
        F.when(
            F.col("qtd_registros_total") > 0,
            F.round((F.col("qtd_registros_com_falha") / F.col("qtd_registros_total")) * 100, 4)
        ).otherwise(F.lit(0.0))
    )
)

display(df_gold_dq_resumo_por_tabela_enderecos)


## 6. KPIs Gold de Endereços — visão geral

In [0]:
df_gold_enderecos_kpis_geral = (
    df_silver_enderecos_flags
    .agg(
        F.lit(NOME_TABELA_SILVER).alias("tabela"),
        F.count("*").cast("int").alias("qtd_enderecos"),
        F.countDistinct("id_cliente").cast("int").alias("qtd_clientes_com_endereco"),
        F.sum(F.when(F.col("linha_limpa"), 1).otherwise(0)).cast("int").alias("qtd_enderecos_limpos"),
        F.sum(F.when(F.col("tem_falha"), 1).otherwise(0)).cast("int").alias("qtd_enderecos_com_falha"),
        F.sum(F.when(F.col("is_principal") == True, 1).otherwise(0)).cast("int").alias("qtd_enderecos_principais"),
        F.countDistinct("bronze_source_file").alias("qtd_arquivos_origem"),
        F.current_timestamp().alias("gold_updated_at")
    )
    .withColumn(
        "pct_enderecos_limpos",
        F.when(
            F.col("qtd_enderecos") > 0,
            F.round((F.col("qtd_enderecos_limpos") / F.col("qtd_enderecos")) * 100, 4)
        ).otherwise(F.lit(0.0))
    )
    .withColumn(
        "pct_enderecos_com_falha",
        F.when(
            F.col("qtd_enderecos") > 0,
            F.round((F.col("qtd_enderecos_com_falha") / F.col("qtd_enderecos")) * 100, 4)
        ).otherwise(F.lit(0.0))
    )
)

display(df_gold_enderecos_kpis_geral)


## 7. KPIs Gold de Endereços por UF

In [0]:
df_gold_enderecos_kpis_por_uf = (
    df_silver_enderecos_flags
    .groupBy("estado")
    .agg(
        F.count("*").cast("int").alias("qtd_enderecos"),
        F.countDistinct("id_cliente").cast("int").alias("qtd_clientes"),
        F.sum(F.when(F.col("linha_limpa"), 1).otherwise(0)).cast("int").alias("qtd_enderecos_limpos"),
        F.sum(F.when(F.col("tem_falha"), 1).otherwise(0)).cast("int").alias("qtd_enderecos_com_falha"),
        F.sum(F.when(F.col("is_principal") == True, 1).otherwise(0)).cast("int").alias("qtd_enderecos_principais"),
        F.current_timestamp().alias("gold_updated_at")
    )
    .withColumn(
        "pct_enderecos_limpos",
        F.when(
            F.col("qtd_enderecos") > 0,
            F.round((F.col("qtd_enderecos_limpos") / F.col("qtd_enderecos")) * 100, 4)
        ).otherwise(F.lit(0.0))
    )
    .withColumn(
        "pct_enderecos_principais",
        F.when(
            F.col("qtd_enderecos") > 0,
            F.round((F.col("qtd_enderecos_principais") / F.col("qtd_enderecos")) * 100, 4)
        ).otherwise(F.lit(0.0))
    )
    .orderBy(F.desc("qtd_enderecos"))
)

display(df_gold_enderecos_kpis_por_uf)


## 8. KPIs Gold de Endereços por Apelido

In [0]:
df_gold_enderecos_kpis_por_apelido = (
    df_silver_enderecos_flags
    .groupBy("apelido")
    .agg(
        F.count("*").cast("int").alias("qtd_enderecos"),
        F.countDistinct("id_cliente").cast("int").alias("qtd_clientes"),
        F.sum(F.when(F.col("linha_limpa"), 1).otherwise(0)).cast("int").alias("qtd_enderecos_limpos"),
        F.sum(F.when(F.col("tem_falha"), 1).otherwise(0)).cast("int").alias("qtd_enderecos_com_falha"),
        F.current_timestamp().alias("gold_updated_at")
    )
    .withColumn(
        "pct_enderecos_limpos",
        F.when(
            F.col("qtd_enderecos") > 0,
            F.round((F.col("qtd_enderecos_limpos") / F.col("qtd_enderecos")) * 100, 4)
        ).otherwise(F.lit(0.0))
    )
    .orderBy(F.desc("qtd_enderecos"))
)

display(df_gold_enderecos_kpis_por_apelido)


## 9. Gravar tabelas Gold em Delta

In [0]:
salvar_delta_gold(
    df_gold_dq_resumo_por_regra_enderecos,
    GOLD_DQ_REGRA_ENDERECOS_TABLE
)

salvar_delta_gold(
    df_gold_dq_resumo_por_tabela_enderecos,
    GOLD_DQ_TABELA_ENDERECOS_TABLE
)

salvar_delta_gold(
    df_gold_enderecos_kpis_geral,
    GOLD_KPI_GERAL_TABLE
)

salvar_delta_gold(
    df_gold_enderecos_kpis_por_uf,
    GOLD_KPI_UF_TABLE
)

salvar_delta_gold(
    df_gold_enderecos_kpis_por_apelido,
    GOLD_KPI_APELIDO_TABLE
)


## 10. Replicar Gold para SQL Server Azure — opcional para Looker

In [0]:
# Esta etapa só executa se as variáveis JDBC estiverem carregadas pelo config.
# Como Gold é agregada/recalculada, usamos overwrite para evitar duplicidade.

replicar_sqlserver_gold(
    df_gold_dq_resumo_por_regra_enderecos,
    "gold_dq_resumo_por_regra_enderecos"
)

replicar_sqlserver_gold(
    df_gold_dq_resumo_por_tabela_enderecos,
    "gold_dq_resumo_por_tabela_enderecos"
)

replicar_sqlserver_gold(
    df_gold_enderecos_kpis_geral,
    "gold_enderecos_kpis_geral"
)

replicar_sqlserver_gold(
    df_gold_enderecos_kpis_por_uf,
    "gold_enderecos_kpis_por_uf"
)

replicar_sqlserver_gold(
    df_gold_enderecos_kpis_por_apelido,
    "gold_enderecos_kpis_por_apelido"
)


## 11. Validação final

In [0]:
tabelas_gold = [
    GOLD_DQ_REGRA_ENDERECOS_TABLE,
    GOLD_DQ_TABELA_ENDERECOS_TABLE,
    GOLD_KPI_GERAL_TABLE,
    GOLD_KPI_UF_TABLE,
    GOLD_KPI_APELIDO_TABLE
]

for tabela in tabelas_gold:
    print("=" * 80)
    print(tabela)
    print("Existe:", spark.catalog.tableExists(tabela))
    print("Registros:", spark.table(tabela).count())
    spark.sql(f"DESCRIBE DETAIL {tabela}").select(
        "format",
        "location",
        "numFiles",
        "sizeInBytes"
    ).show(truncate=False)
